# Lecture 16: Detection + Segmentation

*Notebook for UMich EECS 498/598 (Justin Johnson), Lecture 16 "Detection + Segmentation" (Fall 2020 slides), with material from Stanford CS231n 2026, Lecture 9 "Detection, Segmentation, Visualization, and Understanding".*

Last lecture: **object detection** (a box for every object). Today we finish detection, then go **pixel-level**: label every pixel of the image.

**What you'll learn**
1. How detectors are **trained**: matching proposals to ground-truth boxes.
2. Detection **without anchors** (CornerNet, DETR).
3. **Semantic segmentation** with **fully convolutional networks**, and why they need downsampling and **upsampling**.
4. Every kind of upsampling: unpooling, bilinear, bicubic, max unpooling, and **transposed convolution**.
5. **Instance segmentation** (Mask R-CNN), panoptic segmentation, keypoints, and other "per-region heads".

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## 1. Recap: object detection

| Method | Idea |
|---|---|
| **"Slow" R-CNN** | Run a CNN **separately** on each of ~2,000 cropped regions |
| **Fast R-CNN** | Run the backbone **once**; crop **features** for each region (RoI Pool / RoI Align) |
| **Faster R-CNN** | Also **learn** the region proposals with a Region Proposal Network (anchors) |
| **Single-stage** (YOLO, SSD, RetinaNet) | Each anchor directly predicts the class and box: fully convolutional |

We'll need IoU again:

In [ ]:
def iou(box_a, box_b):
    # Boxes are (x1, y1, x2, y2).
    x1, y1 = max(box_a[0], box_b[0]), max(box_a[1], box_b[1])
    x2, y2 = min(box_a[2], box_b[2]), min(box_a[3], box_b[3])
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    return intersection / (area_a + area_b - intersection)

## 2. How detectors are trained

**R-CNN training:** for each training image, compare every region proposal with the ground-truth (GT) boxes, and put each proposal into one of three groups:
* **positive:** overlaps a GT box well (e.g., IoU > 0.5). Training target: that box's **class** and the **box transform** to it.
* **negative:** barely overlaps anything (e.g., IoU < 0.3). Training target: **"background"** (no box target).
* **neutral:** in between. **Not used** for training (it's ambiguous).

Fast R-CNN is trained the same way, just with cropped features instead of cropped pixels. Faster R-CNN's RPN uses the same idea for its **anchors** (object / not object), with stricter thresholds (positive if IoU > 0.7).

In [ ]:
gt_boxes = {"dog": (20, 30, 120, 130), "cat": (150, 40, 220, 110)}
proposals = [(25, 35, 118, 125), (40, 60, 160, 160), (155, 45, 230, 115), (0, 0, 40, 40), (100, 20, 180, 90)]

for proposal in proposals:
    best_name, best_iou = max(((name, iou(proposal, gt)) for name, gt in gt_boxes.items()), key=lambda item: item[1])
    if best_iou > 0.5:
        group = f"POSITIVE -> class '{best_name}', regress to its box"
    elif best_iou < 0.3:
        group = "NEGATIVE -> class 'background', no box target"
    else:
        group = "NEUTRAL  -> ignored during training"
    print(f"proposal {proposal}: best IoU {best_iou:.2f} ({best_name})  {group}")

**Faster R-CNN's 4 losses** (trained jointly): RPN classification (anchor is object / not), RPN regression (anchor → proposal), object classification (proposal → class or background), object regression (proposal → final box). So a box gets refined **twice**: **anchor → region proposal → object box**.

### Cropping features: why RoI Align matters for training
The cropped features are a function of the image features **and the box coordinates**. RoI Pool "snaps" the box to grid cells, which (1) **misaligns** the features and (2) means we **can't backprop into the box coordinates**. RoI Align samples with **bilinear interpolation** (a weighted average of the 4 nearest grid cells), which fixes both: the features line up with the box, and the whole thing is differentiable.

## 3. Detection without anchors

Everything so far relies on **anchor boxes**. Can we detect objects without them?
* **CornerNet** (Law & Deng, 2018): represent each box by a pair of **corners**. The network predicts a heatmap of upper-left corners and a heatmap of lower-right corners (one per class), plus an "embedding" vector per pixel; corners that belong to the same object are trained to have similar embeddings.
* **DETR** (Carion et al., 2020): a **Transformer** directly outputs a **set** of boxes, matched one-to-one with the ground-truth boxes during training.
  * (+) No RPN! No anchors! No NMS!
  * (−) Slow and brittle to train; at the time, worse than Faster R-CNN on small objects.

## 4. Semantic segmentation

**Label each pixel** of the image with a category (sky, trees, grass, cow, cat…). It **doesn't separate instances**: two cows touching are just "cow pixels". Training data: images where **every pixel** is labeled.

**Idea 1: sliding window.** For each pixel, extract a patch around it and classify the center pixel with a CNN. It works, but it's **very inefficient**: neighboring patches overlap almost completely, and we redo the same computation for each one.

**Idea 2: a fully convolutional network (FCN).** Use **only convolutional layers** (no fully connected layers), so the output has the same height and width as the input, with **C scores per pixel**. Train with **per-pixel cross-entropy**, and predict with an argmax at every pixel.

### Let's build one
Our data: 32×32 images with squares and circles, and a **label for every pixel**: 0 = background, 1 = square, 2 = circle.

In [ ]:
def make_scene(rng, n_shapes=2):
    image = rng.normal(0, 0.05, size=(32, 32))
    labels = np.zeros((32, 32), dtype=int)
    yy, xx = np.mgrid[0:32, 0:32]
    for _ in range(n_shapes):
        size = rng.integers(10, 17)
        x1, y1 = rng.integers(0, 32 - size, size=2)
        if rng.integers(2) == 0:
            region = (xx >= x1) & (xx < x1 + size) & (yy >= y1) & (yy < y1 + size)          # square
            labels[region] = 1
        else:
            cx, cy = x1 + size / 2 - 0.5, y1 + size / 2 - 0.5
            region = (xx - cx)**2 + (yy - cy)**2 <= (size / 2)**2                           # circle
            labels[region] = 2
        image[region] = 1.0 + rng.normal(0, 0.05, region.sum())
    return image, labels

rng = np.random.default_rng(0)
fig, axes = plt.subplots(2, 4, figsize=(9, 4.5))
for col in range(4):
    image, labels = make_scene(rng)
    axes[0, col].imshow(image, cmap="gray"); axes[0, col].set_title("image", fontsize=8)
    axes[1, col].imshow(labels, cmap="viridis", vmin=0, vmax=2); axes[1, col].set_title("label per pixel", fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
plt.show()

**A simple way to write a 3×3 convolution:** a 3×3 filter looks at each pixel's 9 neighbors. So the output is the **sum of 9 shifted copies of the input**, each multiplied by one filter weight (one $C_{out}\times C_{in}$ matrix per shift). That gives a short forward pass and an equally short backward pass:

In [ ]:
def conv3x3_forward(x, W, b):
    # x: (N, C_in, H, W)   W: (C_out, C_in, 3, 3)   b: (C_out,)   ->   (N, C_out, H, W), "same" padding
    N, C_in, H, Wd = x.shape
    padded = np.pad(x, ((0, 0), (0, 0), (1, 1), (1, 1)))
    out = np.zeros((N, W.shape[0], H * Wd))
    for di in range(3):
        for dj in range(3):
            shifted = padded[:, :, di:di + H, dj:dj + Wd].reshape(N, C_in, H * Wd)   # one shifted copy
            out += W[:, :, di, dj] @ shifted                                          # mix its channels
    return out.reshape(N, -1, H, Wd) + b[None, :, None, None]

def conv3x3_backward(d_out, x, W):
    N, C_in, H, Wd = x.shape
    padded = np.pad(x, ((0, 0), (0, 0), (1, 1), (1, 1)))
    d_padded = np.zeros_like(padded)
    dW = np.zeros_like(W)
    d_flat = d_out.reshape(N, W.shape[0], H * Wd)
    for di in range(3):
        for dj in range(3):
            shifted = padded[:, :, di:di + H, dj:dj + Wd].reshape(N, C_in, H * Wd)
            dW[:, :, di, dj] = np.sum(d_flat @ shifted.transpose(0, 2, 1), axis=0)
            d_padded[:, :, di:di + H, dj:dj + Wd] += (W[:, :, di, dj].T @ d_flat).reshape(N, C_in, H, Wd)
    return d_padded[:, :, 1:-1, 1:-1], dW, d_out.sum(axis=(0, 2, 3))

A quick gradient check, as always:

In [ ]:
rng = np.random.default_rng(1)
x_test = rng.normal(size=(2, 3, 5, 5))
W_test, b_test = rng.normal(size=(4, 3, 3, 3)), rng.normal(size=4)
upstream = rng.normal(size=(2, 4, 5, 5))
dx, dW, db = conv3x3_backward(upstream, x_test, W_test)

original = W_test[1, 2, 0, 1]
W_test[1, 2, 0, 1] = original + 1e-6; plus = np.sum(conv3x3_forward(x_test, W_test, b_test) * upstream)
W_test[1, 2, 0, 1] = original - 1e-6; minus = np.sum(conv3x3_forward(x_test, W_test, b_test) * upstream)
W_test[1, 2, 0, 1] = original
print(f"dW check: backprop {dW[1, 2, 0, 1]:.6f}, numerical {(plus - minus) / 2e-6:.6f}")

**The network:** a stack of 3×3 conv layers with ReLU in between; the last layer outputs 3 scores per pixel. The loss is **cross-entropy at every pixel**, averaged.

In [ ]:
def init_fcn(n_layers, channels=8, seed=0):
    rng = np.random.default_rng(seed)
    sizes = [1] + [channels] * (n_layers - 1) + [3]
    return [(rng.normal(0, np.sqrt(2 / (9 * c_in)), size=(c_out, c_in, 3, 3)), np.zeros(c_out))
            for c_in, c_out in zip(sizes[:-1], sizes[1:])]

def fcn_forward(layers, images):
    h = images[:, None]                                   # (N, 1, H, W)
    saved = []
    for i, (W, b) in enumerate(layers):
        z = conv3x3_forward(h, W, b)
        saved.append((h, z))
        h = np.maximum(0, z) if i < len(layers) - 1 else z
    return h, saved                                       # scores: (N, 3, H, W)

def per_pixel_cross_entropy(scores, labels):
    # Softmax over the 3 classes at every pixel, averaged over all pixels.
    shifted = scores - scores.max(axis=1, keepdims=True)
    probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
    N, _, H, W = scores.shape
    n, r, c = np.meshgrid(np.arange(N), np.arange(H), np.arange(W), indexing="ij")
    loss = -np.mean(np.log(probs[n, labels, r, c] + 1e-9))
    probs[n, labels, r, c] -= 1                           # probabilities minus one-hot, at every pixel
    return loss, probs / (N * H * W)

In [ ]:
def train_fcn(n_layers, steps=1500, batch_size=8, lr=0.01, seed=0):
    layers = init_fcn(n_layers, seed=seed)
    m = [(np.zeros_like(W), np.zeros_like(b)) for W, b in layers]
    v = [(np.zeros_like(W), np.zeros_like(b)) for W, b in layers]
    rng = np.random.default_rng(seed)
    for t in range(1, steps + 1):
        scenes = [make_scene(rng) for _ in range(batch_size)]
        images = np.array([s[0] for s in scenes]); labels = np.array([s[1] for s in scenes])
        scores, saved = fcn_forward(layers, images)
        loss, d = per_pixel_cross_entropy(scores, labels)
        for i in reversed(range(len(layers))):            # backward, layer by layer, with an Adam update
            h_in, z = saved[i]
            if i < len(layers) - 1:
                d = d * (z > 0)                            # back through ReLU
            d, dW, db = conv3x3_backward(d, h_in, layers[i][0])
            (W, b), (mW, mb), (vW, vb) = layers[i], m[i], v[i]
            for param, grad, m_, v_ in [(W, dW, mW, vW), (b, db, mb, vb)]:
                m_[...] = 0.9 * m_ + 0.1 * grad
                v_[...] = 0.999 * v_ + 0.001 * grad**2
                param -= lr * (m_ / (1 - 0.9**t)) / (np.sqrt(v_ / (1 - 0.999**t)) + 1e-8)
    return layers

**Problem #1 from the lecture: receptive field.** With $L$ layers of 3×3 convs, each output pixel only sees a $(1 + 2L)\times(1 + 2L)$ patch. For a pixel deep **inside** a big shape, a small patch just looks "bright" everywhere: there's no way to tell a square's interior from a circle's. Let's compare a 3-layer FCN (sees 7×7) with a 6-layer one (sees 13×13), measuring accuracy separately on **interior** pixels (more than 4 pixels from the shape's edge):

In [ ]:
from scipy import ndimage
test_scenes = [make_scene(np.random.default_rng(1000 + i)) for i in range(100)]
test_images = np.array([s[0] for s in test_scenes]); test_labels = np.array([s[1] for s in test_scenes])

# interior pixels: inside a shape, and more than 4 pixels from its edge
interior = np.array([ndimage.binary_erosion(l > 0, iterations=4) for l in test_labels])

for n_layers in [3, 6]:
    layers = train_fcn(n_layers)
    scores, _ = fcn_forward(layers, test_images)
    predictions = scores.argmax(axis=1)
    print(f"{n_layers} conv layers (receptive field {1 + 2 * n_layers}x{1 + 2 * n_layers}): "
          f"all pixels {np.mean(predictions == test_labels):.1%},  interior pixels {np.mean(predictions[interior] == test_labels[interior]):.1%}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
for col, i in enumerate([0, 1]):
    axes[2 * col].imshow(test_labels[i], cmap="viridis", vmin=0, vmax=2); axes[2 * col].set_title("true labels", fontsize=8)
    axes[2 * col + 1].imshow(predictions[i], cmap="viridis", vmin=0, vmax=2); axes[2 * col + 1].set_title("6-layer FCN prediction", fontsize=8)
for ax in axes:
    ax.axis("off")
plt.show()

**What we see:** the deeper network (bigger receptive field) does better, especially in the **interiors**, where context is everything. But getting a big receptive field this way takes many layers, which brings us to **Problem #2: convolutions at full image resolution are expensive.**

**The solution:** design the network with **downsampling** (pooling, strided convolution) in the middle, so later layers see a large area cheaply, and then **upsampling** to get back to full resolution:

**high-res → med-res → low-res → med-res → high-res**

Downsampling we know. **How do we upsample?**

## 5. Upsampling inside the network

### Unpooling: nearest neighbor and "bed of nails"
* **Nearest neighbor:** copy each value into a 2×2 block.
* **Bed of nails:** put each value in the top-left of its 2×2 block and fill the rest with zeros.

In [ ]:
small = np.array([[1, 2],
                  [3, 4]])

nearest = small.repeat(2, axis=0).repeat(2, axis=1)
bed_of_nails = np.zeros((4, 4), dtype=int)
bed_of_nails[::2, ::2] = small
print("nearest neighbor:\n", nearest)
print("bed of nails:\n", bed_of_nails)

### Bilinear interpolation
Each output pixel is a **weighted average of the closest input pixels** in x and y (linear in each direction). To find an output pixel's position in the input, map its **center**: input position = (output position + 0.5) / 2 − 0.5. The lecture's example:

In [ ]:
def bilinear_upsample(image, factor=2):
    H, W = image.shape
    out = np.zeros((H * factor, W * factor))
    for i in range(H * factor):
        for j in range(W * factor):
            y = min(max((i + 0.5) / factor - 0.5, 0), H - 1)      # position in the input (stay inside it)
            x = min(max((j + 0.5) / factor - 0.5, 0), W - 1)
            y0, x0 = int(np.floor(y)), int(np.floor(x))
            y1, x1 = min(y0 + 1, H - 1), min(x0 + 1, W - 1)
            dy, dx = y - y0, x - x0
            out[i, j] = (image[y0, x0] * (1 - dy) * (1 - dx) + image[y0, x1] * (1 - dy) * dx
                         + image[y1, x0] * dy * (1 - dx) + image[y1, x1] * dy * dx)
    return out

print(bilinear_upsample(small.astype(float)))
print("(slides: 1.00 1.25 1.75 2.00 / 1.50 1.75 2.25 2.50 / 2.50 2.75 3.25 3.50 / 3.00 3.25 3.75 4.00)")

### Bicubic interpolation
Uses the **closest 4 pixels in each direction** (16 in total) with a smooth cubic weighting. **This is how we normally resize images.** (The weights below are the standard "cubic convolution" kernel with $a = -0.75$, the same one PyTorch uses.)

In [ ]:
def cubic_weight(d, a=-0.75):
    d = abs(d)
    if d <= 1:
        return (a + 2) * d**3 - (a + 3) * d**2 + 1
    if d < 2:
        return a * d**3 - 5 * a * d**2 + 8 * a * d - 4 * a
    return 0.0

def bicubic_upsample(image, factor=2):
    H, W = image.shape
    out = np.zeros((H * factor, W * factor))
    for i in range(H * factor):
        for j in range(W * factor):
            y, x = (i + 0.5) / factor - 0.5, (j + 0.5) / factor - 0.5
            total = 0.0
            for m in range(int(np.floor(y)) - 1, int(np.floor(y)) + 3):          # 4 rows around the point
                for n in range(int(np.floor(x)) - 1, int(np.floor(x)) + 3):      # 4 columns around the point
                    pixel = image[min(max(m, 0), H - 1), min(max(n, 0), W - 1)]   # repeat edge pixels outside
                    total += pixel * cubic_weight(y - m) * cubic_weight(x - n)
            out[i, j] = total
    return out

print(bicubic_upsample(small.astype(float)).round(2))
print("(slides: 0.68 1.02 1.56 1.89 / 1.35 1.68 2.23 2.56 / 2.44 2.77 3.32 3.65 / 3.11 3.44 3.98 4.32)")

(Notice bicubic can go slightly **outside** the input's range, e.g., 0.68 < 1: that's the cubic curve overshooting a little.)

### Max unpooling
Pair each **max-pooling** layer (on the way down) with a **max-unpooling** layer (on the way up). Max pooling **remembers which position had the max**; max unpooling puts the value back **in that position**, with zeros elsewhere. This helps the upsampled features line up with where things actually were.

In [ ]:
x = np.array([[1, 2, 6, 3],
              [3, 5, 2, 1],
              [1, 2, 2, 1],
              [7, 3, 4, 8]])

pooled = np.zeros((2, 2), dtype=int)
max_positions = {}
for i in range(2):
    for j in range(2):
        block = x[2 * i:2 * i + 2, 2 * j:2 * j + 2]
        r, c = np.unravel_index(np.argmax(block), (2, 2))
        pooled[i, j] = block[r, c]
        max_positions[(i, j)] = (2 * i + r, 2 * j + c)       # remember where the max was

print("max pooled:\n", pooled)

# ... the rest of the network runs here, and changes the values ...
new_values = np.array([[10, 20], [30, 40]])
unpooled = np.zeros((4, 4), dtype=int)
for (i, j), (r, c) in max_positions.items():
    unpooled[r, c] = new_values[i, j]                         # put each value back where its max came from
print("max unpooled (new values in the remembered positions):\n", unpooled)

### Learnable upsampling: transposed convolution
**Recall:** a convolution with **stride 2** moves the filter 2 pixels in the **input** for every 1 pixel in the **output**: it's "learnable downsampling". Can we go the other way?

**Transposed convolution:** for each **input** value, take a copy of the filter **multiplied by that value**, and place it in the output; move **2 pixels in the output** for every 1 pixel in the input; **add up** wherever the copies overlap.

**The lecture's 1-D example:** filter $(x, y, z)$, input $(a, b)$, stride 2:

$$\text{output} = (ax,\ \ ay,\ \ az + bx,\ \ by,\ \ bz)$$

In [ ]:
def transposed_conv1d(inputs, filt, stride=2):
    filt = np.asarray(filt, dtype=float)
    K = len(filt)
    out = np.zeros((len(inputs) - 1) * stride + K)
    for i, value in enumerate(inputs):
        out[i * stride:i * stride + K] += value * filt       # a copy of the filter, weighted by the input
    return out

a, b = 2.0, 3.0
x_, y_, z_ = 1.0, 10.0, 100.0
print("output:", transposed_conv1d([a, b], [x_, y_, z_]))
print("by the formula:", [a * x_, a * y_, a * z_ + b * x_, b * y_, b * z_])

**Why the name "transposed"?** A convolution is a linear operation, so it can be written as **multiplying by a matrix**. A transposed convolution multiplies by **the transpose of that same matrix**. Let's check: build the matrix for a stride-2 convolution, and compare its transpose with our function.

In [ ]:
def conv1d_matrix(input_length, filt, stride=2):
    # The matrix X such that (X @ signal) is the stride-`stride` convolution of `signal` with `filt` (no padding).
    K = len(filt)
    n_out = (input_length - K) // stride + 1
    X = np.zeros((n_out, input_length))
    for i in range(n_out):
        X[i, i * stride:i * stride + K] = filt                # row i: the filter at position i * stride
    return X

filt = np.array([1.0, 10.0, 100.0])
X = conv1d_matrix(5, filt)
print("convolution matrix X (5 inputs -> 2 outputs):\n", X)
print("X.T @ [a, b]          :", X.T @ np.array([a, b]))
print("transposed_conv1d(a, b):", transposed_conv1d([a, b], filt))

They match. (Other names you'll see: upconvolution, fractionally strided convolution, backward strided convolution, and "deconvolution", which is a bad name because it's not the inverse of a convolution.) Fun fact: the transposed convolution is exactly the **backward pass** of a convolution (Lecture 6: "$\partial L/\partial x = W^\top\,\partial L/\partial y$").

**In 2-D:** a 3×3 transposed convolution with stride 2 turns a 2×2 input into a 5×5 output; trimming one pixel from the top and left gives 4×4:

In [ ]:
def transposed_conv2d(inputs, filt, stride=2):
    K = filt.shape[0]
    H, W = inputs.shape
    out = np.zeros(((H - 1) * stride + K, (W - 1) * stride + K))
    for i in range(H):
        for j in range(W):
            out[i * stride:i * stride + K, j * stride:j * stride + K] += inputs[i, j] * filt
    return out

result = transposed_conv2d(np.array([[1.0, 2.0], [3.0, 4.0]]), np.ones((3, 3)))
print("5x5 output:\n", result)
print("trimmed to 4x4:\n", result[1:, 1:])

The overlaps (where copies added up) are visible as the bigger numbers in the middle row and column.

### U-Net (Ronneberger et al., 2015)
A popular segmentation architecture (CS231n 2026):
* a **"downsampling" phase** that increases the field of view but loses spatial detail;
* an **"upsampling" phase** that rebuilds a high-resolution map;
* **skip connections** that **concatenate** each upsampling stage with the high-resolution features from the matching downsampling stage, so fine details aren't lost.

In [ ]:
H = 256
print("U-Net-style shapes (channels x height x width):")
for name, c, s in [("input", 3, H), ("down 1", 64, H), ("down 2", 128, H // 2), ("down 3", 256, H // 4), ("bottom", 512, H // 8),
                   ("up 3: upsample + concat down 3", 512 + 256, H // 4), ("up 2: upsample + concat down 2", 256 + 128, H // 2),
                   ("up 1: upsample + concat down 1", 128 + 64, H), ("output: C class scores per pixel", "C", H)]:
    print(f"  {name:35s} {c} x {s} x {s}")

## 6. Instance segmentation

| | Semantic segmentation | Object detection |
|---|---|---|
| Gives | a label for every pixel | separate object instances |
| Misses | merges instances (two cows = "cow pixels") | only a box, not the shape |

**Things and stuff:**
* **Things:** categories that come in separate instances (cats, cars, people).
* **Stuff:** categories that can't be separated into instances (sky, grass, water, trees).

**Instance segmentation:** detect every object (**things only**) and find **the pixels that belong to each one**.
**Approach:** do object detection, then **predict a segmentation mask for each detected object**.

### Mask R-CNN (He et al., 2017)
Faster R-CNN + a small **mask head**. For each region: RoI Align gives a 256×14×14 feature; a few convs (with upsampling) predict a **28×28 mask for each of the C classes**; we use the mask of the predicted class.

**Training targets:** for a positive proposal, take the ground-truth object's mask, **crop it to the proposal box**, and **resize it to 28×28**:

In [ ]:
image, labels = make_scene(np.random.default_rng(7), n_shapes=1)
object_mask = labels > 0
rows, cols = np.where(object_mask)
gt_box = (cols.min(), rows.min(), cols.max() + 1, rows.max() + 1)
proposal = (gt_box[0] - 3, gt_box[1] - 2, gt_box[2] + 1, gt_box[3] + 3)       # a slightly-off proposal

def mask_target(mask, box, size=28):
    # Crop `mask` to `box` and resize the crop to size x size (nearest neighbor).
    x1, y1, x2, y2 = box
    target = np.zeros((size, size))
    for i in range(size):
        for j in range(size):
            y = int(y1 + (i + 0.5) * (y2 - y1) / size)
            x = int(x1 + (j + 0.5) * (x2 - x1) / size)
            if 0 <= y < mask.shape[0] and 0 <= x < mask.shape[1]:
                target[i, j] = mask[y, x]
    return target

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(image, cmap="gray")
x1, y1, x2, y2 = proposal
axes[0].add_patch(patches.Rectangle((x1 - 0.5, y1 - 0.5), x2 - x1, y2 - y1, fill=False, color="red", lw=2))
axes[0].set_title("image + proposal", fontsize=9)
axes[1].imshow(mask_target(object_mask, proposal), cmap="gray"); axes[1].set_title("28x28 mask target", fontsize=9)
for ax in axes: ax.axis("off")
plt.show()

The mask head is trained with a **per-pixel binary cross-entropy** loss on these 28×28 targets. Mask R-CNN gives very good results, and the same model can also predict human pose.

## 7. Beyond instance segmentation

### Panoptic segmentation
**Label every pixel** (things **and** stuff), and for "thing" categories, **also separate the instances**: "tree, tree, grass, cow #1, cow #2, …".

### Human keypoints
Represent a person's pose by **17 keypoints**: nose, left/right eye, ear, shoulder, elbow, wrist, hip, knee, ankle. **Mask R-CNN can do this too:** add a head that predicts one **56×56 "mask" per keypoint**, where the ground truth has exactly **one pixel turned on** (the keypoint's location). Train it with a **softmax over all 56×56 positions**, so the network learns *where* each keypoint is:

In [ ]:
target_position = (20, 33)                                 # the true location of, say, the left ankle
scores = np.random.default_rng(3).normal(0, 1, size=(56, 56))
scores[20, 33] += 6.0                                      # a network that is fairly confident about the right spot

probs = np.exp(scores - scores.max())
probs /= probs.sum()                                       # softmax over all 3,136 positions
loss = -np.log(probs[target_position])
predicted = tuple(int(v) for v in np.unravel_index(np.argmax(probs), probs.shape))
print(f"predicted keypoint location: {predicted}, true: {target_position}, loss: {loss:.2f}")
print(f"(a completely uniform guess would have loss log(3136) = {np.log(56 * 56):.2f})")

### The general idea: add per-region "heads" to Faster / Mask R-CNN
Each head receives the features after RoI Pool / RoI Align and makes some prediction **per region**:
* **Mask head:** instance segmentation (Mask R-CNN).
* **Keypoint head:** pose estimation.
* **Caption head** (an LSTM, Lecture 12): **dense captioning**, a sentence for every region (Johnson, Karpathy & Fei-Fei, "DenseCap", 2016).
* **Mesh head:** a **3-D triangle mesh** for every object (Gkioxari, Malik & Johnson, "Mesh R-CNN", 2019). More in the 3-D vision lecture.

## 8. In practice (optional, run in Colab)

In [ ]:
# [PyTorch] Needs: pip install torch torchvision (downloads pretrained weights)
import torch, torch.nn as nn, torch.nn.functional as F, torchvision

small = torch.tensor([[[[1.0, 2.0], [3.0, 4.0]]]])
print(F.interpolate(small, scale_factor=2, mode="bilinear", align_corners=False))   # same numbers as our bilinear
print(F.interpolate(small, scale_factor=2, mode="bicubic", align_corners=False))    # same numbers as our bicubic

up = nn.ConvTranspose2d(in_channels=64, out_channels=32, kernel_size=3, stride=2, padding=1, output_padding=1)
print("transposed conv:", tuple(up(torch.randn(1, 64, 14, 14)).shape))              # 14x14 -> 28x28

mask_rcnn = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT").eval()
with torch.no_grad():
    out = mask_rcnn([torch.rand(3, 480, 640)])[0]                                   # use a real photo here
print("boxes", out["boxes"].shape, " masks", out["masks"].shape)
print("RoI Align:", torchvision.ops.roi_align)

## Summary

| Task | What it outputs | Typical method |
|---|---|---|
| **Classification** | one label | CNN (or ViT) |
| **Semantic segmentation** | a label per pixel (things + stuff, no instances) | fully convolutional net with down/upsampling (FCN, U-Net) |
| **Object detection** | a class + box per object | Faster R-CNN (two-stage), YOLO/SSD/RetinaNet (single-stage), DETR |
| **Instance segmentation** | a class + mask per object (things) | Mask R-CNN |
| **Panoptic segmentation** | every pixel labeled, instances separated | Panoptic FPN |
| **Keypoints** | locations of body parts | Mask R-CNN with a keypoint head |

* Detectors are trained by **matching proposals to ground truth** (positive / negative / neutral).
* **Fully convolutional networks** predict all pixels at once; **downsampling** grows the receptive field cheaply, **upsampling** restores resolution.
* **Upsampling:** nearest, bed of nails, bilinear, bicubic, max unpooling (fixed), or **transposed convolution** (learnable).

### Check your understanding
1. Why is the sliding-window approach to segmentation so slow?
2. Why does a segmentation network need a big receptive field? What's the cheap way to get one?
3. What does a transposed convolution with stride 2 do, and why is it called "transposed"?
4. What's the difference between semantic and instance segmentation? Which handles "stuff"?
5. How does Mask R-CNN predict keypoints?

<details><summary>Answers</summary>

1. It runs a CNN on an overlapping patch for every pixel, repeating almost all the computation.
2. To tell what a pixel belongs to, it needs context (e.g., the interior of a square vs. a circle looks the same up close). Downsampling inside the network makes later layers see a large area cheaply.
3. It upsamples 2× by placing filter copies weighted by each input value and summing overlaps. It's multiplication by the transpose of the matrix of a regular (stride-2) convolution.
4. Semantic labels pixels but merges instances (handles things and stuff); instance segmentation separates object instances but only handles things.
5. One 56×56 heatmap per keypoint, trained with a softmax over all positions against a one-hot target.
</details>

**Next:** 3-D vision.